# Qwen3.8-9B Recurrent-Loop (`Base` + `K=1..6`) — Precompiled Colab T4 + OpenCode Agentic Demo

Run **Qwen3.8-9B** with **1 to 6 internal reasoning loops** (`152` virtual layers at `K=6` reusing the same `32` quantized blocks) on a free Google Colab **T4 GPU (16 GB VRAM)** — **no compilation needed** (~5 seconds setup), plus full **OpenCode** terminal setup so you can test **agentic coding** across the base model and every loop depth (`1..6`) immediately!

- **Engine**: [`CympUser/llama.cpp-qwen-loop`](https://github.com/CympUser/llama.cpp-qwen-loop)
- **Base Model**: [`mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF`](https://huggingface.co/mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF) (`Q4_K_S`)
- **Loop-6 Adapter (`45.27 MB`)**: [`Vnswz/Qwen3.8-9B-Loop6-Adapter`](https://huggingface.co/Vnswz/Qwen3.8-9B-Loop6-Adapter)

> Make sure your Colab runtime is set to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
# 1. Check GPU, ensure CUDA 12 runtime libs, & download precompiled Linux CUDA binary (~5s)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!mkdir -p /content/qwen-loop/models
%cd /content/qwen-loop

!wget -q --show-progress -O llama-linux-cuda.tar.gz \
  https://github.com/CympUser/llama.cpp-qwen-loop/releases/download/v1.0.0/llama-cpp-qwen-loop-v1.0.0-linux-cuda-x64.tar.gz
!tar -xzf llama-linux-cuda.tar.gz && chmod +x llama-cli llama-server

# Locate libcudart.so.12 & libcublas.so.12 (install lightweight pip wheels if Colab lacks CUDA 12 libs)
import glob, os, subprocess, sys

def find_cuda12_lib_dirs():
    dirs = set()
    patterns = [
        "/usr/local/cuda-12*/lib64/libcudart.so.12*",
        "/usr/local/cuda-12*/lib64/libcublas.so.12*",
        "/usr/local/cuda/lib64/libcudart.so.12*",
        "/usr/local/cuda/lib64/libcublas.so.12*",
        "/usr/local/lib/python*/dist-packages/nvidia/*/lib/libcudart.so.12*",
        "/usr/local/lib/python*/dist-packages/nvidia/*/lib/libcublas.so.12*",
        "/usr/lib/x86_64-linux-gnu/libcudart.so.12*",
        "/usr/lib/x86_64-linux-gnu/libcublas.so.12*",
    ]
    for pat in patterns:
        for p in glob.glob(pat):
            dirs.add(os.path.dirname(p))
    return sorted(dirs)

lib_dirs = find_cuda12_lib_dirs()
has_cudart = any(os.path.exists(os.path.join(d, "libcudart.so.12")) for d in lib_dirs)
has_cublas = any(os.path.exists(os.path.join(d, "libcublas.so.12")) for d in lib_dirs)
if not (has_cudart and has_cublas):
    print("Installing lightweight CUDA 12 runtime libs (nvidia-cuda-runtime-cu12, nvidia-cublas-cu12)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "nvidia-cuda-runtime-cu12", "nvidia-cublas-cu12"])
    lib_dirs = find_cuda12_lib_dirs()

os.environ["LD_LIBRARY_PATH"] = ":".join(lib_dirs + [os.environ.get("LD_LIBRARY_PATH", "")])
print("CUDA 12 lib dirs:", lib_dirs)
!./llama-cli --version

In [ ]:
# 2. Download Base GGUF (5.23 GB) and Loop-6 Adapter (45 MB)
!wget -q --show-progress -O models/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf \
  https://huggingface.co/mradermacher/Qwen3.8-9B-Distill-uncensored-heretic-GGUF/resolve/main/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf
!wget -q --show-progress -O models/qwen3.8-9b-heretic-loop6-lora.gguf \
  https://huggingface.co/Vnswz/Qwen3.8-9B-Loop6-Adapter/resolve/main/qwen3.8-9b-heretic-loop6-lora.gguf
!ls -lh models/

In [ ]:
# 3. Helper to run the exact same prompt at any loop depth K in 1..6 (via llama-cli)
import os, subprocess

def run_qwen_loop(prompt: str, num_loops: int = 6, max_tokens: int = 512) -> str:
    cmd = [
        "./llama-cli",
        "-m", "models/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf",
        "--lora", "models/qwen3.8-9b-heretic-loop6-lora.gguf",
        "--override-kv", f"qwen35.num_loops=int:{num_loops},qwen35.recurrent_kv_cache_mode=str:per_loop",
        "-c", "8192", "-ngl", "999", "--fit", "off",
        "--cache-type-k", "q8_0", "--cache-type-v", "q8_0",
        "--temp", "0.0", "-n", str(max_tokens),
        "-st", "--simple-io",
        "-p", prompt,
    ]
    res = subprocess.run(cmd, capture_output=True, text=True, env=os.environ.copy())
    if res.returncode != 0:
        print(res.stderr)
    return res.stdout

In [ ]:
# 4. Side-by-side comparison: 4-Item Deductive Ordering (K=1 vs K=6)
logic_prompt = (
    "Four servers (Alpha, Beta, Gamma, Delta) are ranked from fastest to slowest. "
    "Clues: (1) Beta is faster than Gamma. (2) Alpha is slower than Gamma. "
    "(3) Delta is faster than Beta. "
    "Reply ONLY with the final order from fastest to slowest using '>' (e.g., A>B>C>D)."
)

print("=== K=1 (Base single-pass) ===")
print(run_qwen_loop(logic_prompt, num_loops=1, max_tokens=384))

print("=== K=6 (6 Internal Reasoning Loops) ===")
print(run_qwen_loop(logic_prompt, num_loops=6, max_tokens=384))

In [ ]:
# 5. Try your own prompt! Change NUM_LOOPS between 1 and 6
MY_PROMPT = "Explain in 3 concise bullet points why recurrent-depth transformers save VRAM compared to deeper models."
NUM_LOOPS = 6  # Try 1, 2, 4, or 6

print(run_qwen_loop(MY_PROMPT, num_loops=NUM_LOOPS, max_tokens=512))

---
## Stage 6 — Agentic Terminal Mode with **OpenCode** (`Base` + `Loop 1..6`)

Run the cell below to:
1. Install **OpenCode** CLI and expose it globally (`/usr/local/bin/opencode`).
2. Launch `llama-server` in the background on `http://127.0.0.1:8089/v1` with **all 7 model presets**:
   - `llamacpp/qwen3.8-9b-base` *(original Qwen3.8-9B without LoRA)*
   - `llamacpp/qwen3.8-9b-loop1` through `llamacpp/qwen3.8-9b-loop6` *(default: `loop6`)*
3. Configure `~/.config/opencode/opencode.json` automatically for the installed OpenCode version.

Then open the **Colab Terminal** (bottom-left `>_` icon) and type `opencode` — use `/models` inside OpenCode to switch between **Base** and **Loop 1..6** live!

In [ ]:
# 6. Install OpenCode, configure Base + Loop 1..6 presets, and start llama-server on port 8089
import json, os, shutil, subprocess, time, urllib.request
from pathlib import Path

SERVER_PORT = 8089  # Avoids Colab's built-in daemon on port 8080
WORK = Path("/content/qwen-loop")
BASE_MODEL = WORK / "models/Qwen3.8-9B-Distill-uncensored-heretic.Q4_K_S.gguf"
LORA_MODEL = WORK / "models/qwen3.8-9b-heretic-loop6-lora.gguf"

# 6a. Install OpenCode CLI if not already installed
opencode_bin = shutil.which("opencode")
if not opencode_bin:
    print("[1/4] Installing OpenCode CLI...")
    subprocess.run("curl -fsSL https://opencode.ai/install | bash", shell=True, check=True)
    for cand in [
        Path.home() / ".opencode/bin/opencode",
        Path.home() / ".local/bin/opencode",
        Path("/usr/local/bin/opencode"),
    ]:
        if cand.exists():
            opencode_bin = str(cand)
            if str(cand) != "/usr/local/bin/opencode":
                os.system(f"ln -sf {cand} /usr/local/bin/opencode")
            break
if not opencode_bin:
    raise RuntimeError("OpenCode binary not found after installation.")

ver_out = subprocess.check_output([opencode_bin, "--version"], text=True).strip()
is_v2 = " 2." in ver_out or "v2." in ver_out or ver_out.startswith("2.")
print(f"[OK] OpenCode installed: {ver_out} (schema={'v2' if is_v2 else 'v1'})")

# 6b. Write models_colab.ini with Base + Loop 1..6 (T4 16 GB VRAM -> ctx-size = 16384)
ini_lines = [
    "[*]",
    "n-gpu-layers = 999",
    "fit = off",
    "cache-type-k = q8_0",
    "cache-type-v = q8_0",
    "repeat-penalty = 1.15",
    "ctx-size = 16384",
    "parallel = 1",
    "cache-ram = 1024",
    "ctx-checkpoints = 2",
    "",
    "[qwen3.8-9b-base]",
    f"model = {BASE_MODEL}",
    "",
]
for k in range(1, 7):
    ini_lines += [
        f"[qwen3.8-9b-loop{k}]",
        f"model = {BASE_MODEL}",
        f"lora = {LORA_MODEL}",
        f"override-kv = qwen35.num_loops=int:{k},qwen35.recurrent_kv_cache_mode=str:per_loop",
        "",
    ]
ini_path = WORK / "models_colab.ini"
ini_path.write_text("\n".join(ini_lines), encoding="utf-8")
print(f"[2/4] Wrote {ini_path} with 7 presets (base + loop1..loop6)")

# 6c. Write ~/.config/opencode/opencode.json matching installed OpenCode version
model_specs = [
    ("qwen3.8-9b-loop6", "Qwen 3.8 9B Heretic (6-Loop - Max Quality)"),
    ("qwen3.8-9b-loop5", "Qwen 3.8 9B Heretic (5-Loop)"),
    ("qwen3.8-9b-loop4", "Qwen 3.8 9B Heretic (4-Loop - Balanced)"),
    ("qwen3.8-9b-loop3", "Qwen 3.8 9B Heretic (3-Loop)"),
    ("qwen3.8-9b-loop2", "Qwen 3.8 9B Heretic (2-Loop - Fast)"),
    ("qwen3.8-9b-loop1", "Qwen 3.8 9B Heretic (1-Loop Adapter)"),
    ("qwen3.8-9b-base",  "Qwen 3.8 9B Heretic (Base - No Adapter)"),
]
if is_v2:
    models_cfg = {
        mid: {
            "modelID": mid,
            "name": label,
            "capabilities": {"tools": True, "reasoning": True, "input": ["text"], "output": ["text"]},
            "limit": {"context": 16384, "output": 4096},
        }
        for mid, label in model_specs
    }
    opencode_cfg = {
        "$schema": "https://opencode.ai/config.json",
        "model": "llamacpp/qwen3.8-9b-loop6",
        "small_model": "llamacpp/qwen3.8-9b-base",
        "providers": {
            "llamacpp": {
                "name": "llama.cpp Qwen-Loop (Colab T4)",
                "package": "@opencode/ai/providers/openai-compatible",
                "settings": {
                    "baseURL": f"http://127.0.0.1:{SERVER_PORT}/v1",
                    "apiKey": "local",
                    "headerTimeout": 120000,
                    "chunkTimeout": 600000,
                },
                "models": models_cfg,
            }
        },
    }
else:
    models_cfg = {
        mid: {"name": label, "limit": {"context": 16384, "output": 4096}}
        for mid, label in model_specs
    }
    opencode_cfg = {
        "$schema": "https://opencode.ai/config.json",
        "model": "llamacpp/qwen3.8-9b-loop6",
        "small_model": "llamacpp/qwen3.8-9b-base",
        "provider": {
            "llamacpp": {
                "npm": "@ai-sdk/openai-compatible",
                "name": "llama.cpp Qwen-Loop (Colab T4)",
                "options": {"baseURL": f"http://127.0.0.1:{SERVER_PORT}/v1", "apiKey": "local"},
                "models": models_cfg,
            }
        },
    }

cfg_dir = Path.home() / ".config/opencode"
cfg_dir.mkdir(parents=True, exist_ok=True)
cfg_path = cfg_dir / "opencode.json"
cfg_path.write_text(json.dumps(opencode_cfg, indent=2), encoding="utf-8")
print(f"[3/4] Wrote {cfg_path}")

# 6d. Start llama-server in background (stop any previous instance first)
subprocess.run(["pkill", "-f", "llama-server"], stderr=subprocess.DEVNULL)
time.sleep(1)
log_f = open(WORK / "llama_server.log", "w", encoding="utf-8")
srv_proc = subprocess.Popen(
    [str(WORK / "llama-server"), "--models-preset", str(ini_path), "--port", str(SERVER_PORT), "--jinja"],
    cwd=str(WORK),
    env=os.environ.copy(),
    stdout=log_f,
    stderr=subprocess.STDOUT,
)

ready = False
for _ in range(45):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{SERVER_PORT}/health", timeout=2) as r:
            if r.status == 200:
                ready = True
                break
    except Exception:
        time.sleep(1)

if not ready:
    print((WORK / "llama_server.log").read_text(encoding="utf-8")[-2000:])
    raise RuntimeError(f"llama-server failed to start on port {SERVER_PORT}")

with urllib.request.urlopen(f"http://127.0.0.1:{SERVER_PORT}/v1/models", timeout=5) as r:
    srv_models = [m["id"] for m in json.loads(r.read())["data"]]

print(f"[4/4] llama-server is LIVE on http://127.0.0.1:{SERVER_PORT}/v1 (PID={srv_proc.pid})")
print("Available models:", srv_models)
print("\n" + "="*66)
print(" READY! Open the Colab Terminal (>_ icon) and run:  opencode")
print(" Inside OpenCode, type /models to switch between Base & Loop 1..6!")
print("="*66)

In [ ]:
# 7. Optional: test OpenCode directly from this cell (change loop6 -> base, loop1..loop6)
!opencode models | grep llamacpp
!opencode run -m llamacpp/qwen3.8-9b-loop6 "Create a file fib.py that prints the first 10 Fibonacci numbers, then run it and show the output."